# ChurnLens — 02. Exploratory Data Analysis (EDA)
## Multi-Dimensional Churn, Cohort & Revenue Deep-Dive

### Analytical Focus:
1. **Overall Churn & Revenue Exposure**: Monthly Recurring Revenue (MRR) Lost vs Preserved
2. **Temporal & Cohort Dynamics**: Retention decay curves and tenure cliff analysis
3. **Plan & Segment Distribution**: Where is revenue concentration and churn risk?
4. **Acquisition Channel Quality**: High-intent organic vs low-intent paid churn disparities
5. **Behavioral Interaction Patterns**: Login frequency, session duration, and feature adoption


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set aesthetics
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['font.size'] = 10
plt.rcParams['figure.dpi'] = 120

df = pd.read_csv('../data/cleaned_churn_data.csv')
print(f"Loaded {len(df):,} customers.")


### 1. Overall Churn Rate & Revenue Loss Summary


In [ ]:
total_cust = len(df)
churn_cust = df['churned'].sum()
churn_rate = df['churned'].mean() * 100
total_mrr = df['monthly_spend'].sum()
lost_mrr = df[df['churned'] == 1]['monthly_spend'].sum()
rev_churn_rate = (lost_mrr / total_mrr) * 100

summary_df = pd.DataFrame({
    'Metric': [
        'Total Customer Base', 'Active Customers', 'Churned Customers', 
        'Customer Churn Rate (%)', 'Total Potential MRR ($)', 'Lost MRR to Churn ($)', 'Revenue Churn Rate (%)'
    ],
    'Value': [
        f"{total_cust:,}", f"{total_cust - churn_cust:,}", f"{churn_cust:,}",
        f"{churn_rate:.2f}%", f"${total_mrr:,.2f}", f"${lost_mrr:,.2f}", f"{rev_churn_rate:.2f}%"
    ]
})
summary_df


### 2. Churn Rate by Customer Segment & Subscription Plan


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Segment churn
seg_order = ['Enterprise', 'Mid-Market', 'SMB', 'Startup/Individual']
sns.barplot(
    data=df, x='customer_segment', y='churned', order=seg_order,
    palette='Blues_r', ax=axes[0], errorbar=None
)
axes[0].set_title('Churn Rate by Customer Segment', fontsize=12, fontweight='bold', pad=10)
axes[0].set_ylabel('Churn Rate (%)')
axes[0].set_xlabel('')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y*100:.0f}%'))

# Plan churn
plan_order = ['Custom Tier', 'Enterprise', 'Pro', 'Basic']
sns.barplot(
    data=df, x='subscription_plan', y='churned', order=plan_order,
    palette='Purples_r', ax=axes[1], errorbar=None
)
axes[1].set_title('Churn Rate by Subscription Plan', fontsize=12, fontweight='bold', pad=10)
axes[1].set_ylabel('Churn Rate (%)')
axes[1].set_xlabel('')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y*100:.0f}%'))

plt.tight_layout()
plt.show()


### 3. Customer Tenure & The Onboarding Cliff
Analysis of churn likelihood across customer tenure brackets.


In [ ]:
# Tenure Bracket Analysis
bins = [0, 3, 6, 12, 18, 24]
labels = ['1-3 Months', '4-6 Months', '7-12 Months', '13-18 Months', '19-24 Months']
df['tenure_bracket'] = pd.cut(df['tenure_months'], bins=bins, labels=labels)

tenure_agg = df.groupby('tenure_bracket', observed=False).agg(
    total_customers=('customer_id', 'count'),
    churned_customers=('churned', 'sum'),
    churn_rate=('churned', 'mean'),
    avg_mrr=('monthly_spend', 'mean')
).reset_index()

tenure_agg['churn_rate_pct'] = (tenure_agg['churn_rate'] * 100).round(2)
tenure_agg


In [ ]:
plt.figure(figsize=(10, 4.5))
ax = sns.barplot(data=tenure_agg, x='tenure_bracket', y='churn_rate_pct', palette='viridis')
plt.title('Churn Rate by Customer Tenure (The Onboarding Cliff)', fontsize=12, fontweight='bold', pad=12)
plt.ylabel('Churn Rate (%)')
plt.xlabel('Customer Tenure')
for p in ax.patches:
    ax.annotate(f"{p.get_height():.1f}%", (p.get_x() + p.get_width() / 2., p.get_height() / 2),
                ha='center', va='center', color='white', fontweight='bold')
plt.tight_layout()
plt.show()


### 4. Acquisition Channel Retention Quality & LTV


In [ ]:
channel_summary = df.groupby('acquisition_channel').agg(
    total_acquired=('customer_id', 'count'),
    churn_rate=('churned', 'mean'),
    avg_monthly_spend=('monthly_spend', 'mean'),
    avg_total_ltv=('total_spend', 'mean'),
    key_feature_adoption=('key_feature_usage', 'mean')
).reset_index().sort_values(by='churn_rate')

channel_summary['churn_rate_pct'] = (channel_summary['churn_rate'] * 100).round(2)
channel_summary['key_feature_adoption_pct'] = (channel_summary['key_feature_adoption'] * 100).round(1)
channel_summary[['acquisition_channel', 'total_acquired', 'churn_rate_pct', 'avg_total_ltv', 'key_feature_adoption_pct']]


### 5. Behavioral Usage: Login Recency & Monthly Sessions


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Days inactive distribution
sns.kdeplot(data=df, x='days_since_last_login', hue='churned', common_norm=False, fill=True, palette=['#2b5c8f', '#d9534f'], ax=axes[0])
axes[0].set_title('Days Inactive Distribution by Churn Status', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Days Since Last Login')
axes[0].legend(['Churned', 'Active'])

# 30-day session count
sns.boxplot(data=df, x='churned', y='sessions_last_30_days', palette=['#2b5c8f', '#d9534f'], ax=axes[1])
axes[1].set_title('Sessions (Last 30 Days) by Churn Status', fontsize=11, fontweight='bold')
axes[1].set_xticklabels(['Active (0)', 'Churned (1)'])
axes[1].set_ylabel('Sessions Last 30 Days')

plt.tight_layout()
plt.show()
